# Travail demandé — Étude et expérimentation de 29 algorithmes de Machine Learning

Ce notebook contient **l'intégralité des réponses** exigées : Q1 à Q4 pour chacun des 29 algorithmes, ainsi que les parties Q0 (visualisations) et Q5 à Q8 (expérimentation finale).

In [ ]:
!pip -q install scikit-learn xgboost umap-learn gymnasium

In [ ]:
# Jeux publics de scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score, silhouette_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
GRAINE = 42

jeu_reg = load_diabetes(as_frame=True)
X_reg, y_reg = jeu_reg.data.values, jeu_reg.target.values
jeu_cls = load_breast_cancer(as_frame=True)
X_cls, y_cls = jeu_cls.data.values, jeu_cls.target.values
jeu_groupes = load_wine(as_frame=True)
X_clu = PCA(n_components=2, random_state=GRAINE).fit_transform(StandardScaler().fit_transform(jeu_groupes.data.values))
jeu_images = load_digits()
X_images = StandardScaler().fit_transform(jeu_images.data)
Xr_ent, Xr_test, yr_ent, yr_test = train_test_split(X_reg,y_reg,test_size=.25,random_state=GRAINE)
Xc_ent, Xc_test, yc_ent, yc_test = train_test_split(X_cls,y_cls,test_size=.25,stratify=y_cls,random_state=GRAINE)
print('Diabetes',X_reg.shape,'Breast cancer',X_cls.shape,'Wine',X_clu.shape,'Digits',X_images.shape)

## Q0 : Visualisation et exploration des données de base

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.scatterplot(x=jeu_reg.data['bmi'], y=jeu_reg.target, color='blue', alpha=0.6, ax=axes[0, 0])
axes[0, 0].set_title("Régression (Diabetes)")
sns.kdeplot(x=jeu_cls.data['mean radius'], hue=jeu_cls.target, fill=True, ax=axes[0, 1], legend=False)
axes[0, 1].set_title("Classification (Cancer) : Distribution")
sns.scatterplot(x=jeu_groupes.data['alcohol'], y=jeu_groupes.data['flavanoids'], hue=jeu_groupes.target, palette='viridis', ax=axes[1, 0], legend=False)
axes[1, 0].set_title("Clustering (Wine)")
axes[1, 1].imshow(jeu_images.images[0], cmap='gray')
axes[1, 1].set_title(f"Image (Digits) Cible: {jeu_images.target[0]}")
plt.tight_layout(); plt.show()

## 1. Régression linéaire

### Présentation (Q1)
**Nom :** Régression Linéaire (Linear Regression). **Famille :** Régression.
**Intuition :** Trouver l'hyperplan (ou droite) minimisant la somme des carrés des écarts entre prédictions et valeurs réelles.
**Formule :** $\hat{y} = w^T X + b$ ($w$: poids, $b$: biais). Les paramètres sont appris. Pas d'hyperparamètre.
**Avantages :** 1. Très interprétable. 2. Entraînement rapide.
**Limites :** 1. Ne capte que des relations linéaires. 2. Sensible aux outliers.

In [ ]:
plt.figure(figsize=(4,3)); np.random.seed(0); x=np.linspace(0,10,20); y=x+np.random.normal(0,1,20); plt.scatter(x,y, color='blue'); plt.plot(x,x, color='red'); plt.title('Schéma Algo 1'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Finance) :** Prédiction des ventes selon le budget pub.
**Cas 2 (Logistique) :** Temps de trajet selon la distance.
**Inapproprié :** Données à forte saisonnalité ou cycliques (ex: météo).
**Métriques :** RMSE, MAE, R².

In [ ]:
from sklearn.linear_model import LinearRegression
modele=LinearRegression().fit(Xr_ent,yr_ent)
pred=modele.predict(Xr_test)
print('RMSE:',mean_squared_error(yr_test,pred)**.5,'R²:',r2_score(yr_test,pred))

### Analyse et variation d'hyperparamètre (Q4)
Le R² de ~0.49 montre que le modèle explique presque la moitié de la variance. La relation médicale n'est pas strictement linéaire. **Variation :** Si on normalise les données avant (StandardScaler), le R² reste identique, mais les coefficients $w$ deviennent comparables entre eux.

## 2. Ridge

### Présentation (Q1)
**Nom :** Régression Ridge (Ridge Regression). **Famille :** Régression.
**Intuition :** Régression linéaire ajoutant une pénalité L2 pour forcer les poids à rester petits et éviter le surapprentissage.
**Formule :** $\min ||y - Xw||^2_2 + \alpha ||w||^2_2$. Paramètres: $w$. Hyperparamètre: $\alpha$ (pénalité).
**Avantages :** 1. Réduit l'overfitting. 2. Gère la multicollinéarité.
**Limites :** 1. Ne met aucun poids exactement à zéro. 2. Nécessite la standardisation des données.

In [ ]:
plt.figure(figsize=(4,3)); np.random.seed(0); x=np.linspace(0,10,20); y=x+np.random.normal(0,1,20); plt.scatter(x,y, color='blue'); plt.plot(x,x, color='red'); plt.title('Schéma Algo 2'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Gestion) :** Modélisation de risques de crédit avec beaucoup d'indicateurs corrélés.
**Cas 2 (Marketing) :** Prédiction de la valeur à vie (LTV) d'un client.
**Inapproprié :** Si l'on veut un modèle parcimonieux (sélection de variables).
**Métriques :** MAE, RMSE, R².

In [ ]:
from sklearn.linear_model import Ridge
modele=make_pipeline(StandardScaler(),Ridge(alpha=25.)).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))

### Analyse et variation d'hyperparamètre (Q4)
L'Erreur Absolue Moyenne (MAE) donne l'erreur moyenne en unités cibles (progression de la maladie). **Variation :** Si on augmente fortement l'hyperparamètre `alpha` (ex: alpha=1000), les coefficients s'écrasent vers zéro, augmentant le biais et dégradant la MAE.

## 3. Lasso

### Présentation (Q1)
**Nom :** Lasso (Least Absolute Shrinkage and Selection Operator). **Famille :** Régression.
**Intuition :** Régression avec pénalité L1 qui force certains poids à devenir exactement zéro (sélection de variables).
**Formule :** $\min ||y - Xw||^2_2 + \alpha ||w||_1$. Hyperparamètre: $\alpha$.
**Avantages :** 1. Effectue une sélection automatique de features. 2. Modèles plus simples/parcimonieux.
**Limites :** 1. Peut exclure des variables corrélées utiles. 2. Plus lent à optimiser que Ridge.

In [ ]:
plt.figure(figsize=(4,3)); np.random.seed(0); x=np.linspace(0,10,20); y=x+np.random.normal(0,1,20); plt.scatter(x,y, color='blue'); plt.plot(x,x, color='red'); plt.title('Schéma Algo 3'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Commerce) :** Déterminer les 3 facteurs principaux impactant le CA parmi 50 variables.
**Cas 2 (Biologie) :** Identifier un petit sous-ensemble de gènes actifs.
**Inapproprié :** Quand toutes les variables ont un impact dense et utile.
**Métriques :** MAE, RMSE, R².

In [ ]:
from sklearn.linear_model import Lasso
modele=make_pipeline(StandardScaler(),Lasso(alpha=3.,max_iter=15000)).fit(Xr_ent,yr_ent)
print('R²:',r2_score(yr_test,modele.predict(Xr_test)))

### Analyse et variation d'hyperparamètre (Q4)
Le R² montre la qualité de l'ajustement. **Variation :** En augmentant `alpha`, de plus en plus de variables sont annulées. Si on affiche `modele.named_steps['lasso'].coef_`, on constatera que la pénalité L1 a effectivement mis certains coefficients à zéro.

## 4. Régression polynomiale

### Présentation (Q1)
**Nom :** Régression polynomiale. **Famille :** Régression.
**Intuition :** Ajouter des caractéristiques artificielles (puissances et interactions, ex: $x^2, x_1 x_2$) avant d'appliquer un modèle linéaire.
**Formule :** $\hat{y} = w_0 + w_1 x + w_2 x^2 + ...$ Hyperparamètre: `degree`.
**Avantages :** 1. Capte des courbures non linéaires. 2. Utilise les outils linéaires standards.
**Limites :** 1. Explosion combinatoire du nombre de features. 2. Surapprentissage rapide si degré élevé.

In [ ]:
plt.figure(figsize=(4,3)); np.random.seed(0); x=np.linspace(-3,3,30); y=x**2+np.random.normal(0,1,30); plt.scatter(x,y, color='blue'); plt.plot(np.sort(x), np.sort(x)**2, color='red'); plt.title('Schéma Algo 4'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Finance) :** Modélisation des coûts marginaux ou d'économies d'échelle.
**Cas 2 (Physique) :** Trajectoire d'un objet.
**Inapproprié :** Avec beaucoup de variables initiales (le nombre de termes polynomiaux exploserait).
**Métriques :** MAE, RMSE.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
modele=make_pipeline(StandardScaler(),PolynomialFeatures(degree=2,include_bias=False),Ridge(alpha=60.)).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))

### Analyse et variation d'hyperparamètre (Q4)
L'ajout de caractéristiques polynomiales permet de réduire la MAE par rapport au modèle linéaire simple si la relation est courbe. **Variation :** Passer le `degree` à 3 ferait chuter les performances sur l'ensemble de test à cause d'un fort surapprentissage (fléau de la dimension).

## 5. Arbre de régression

### Présentation (Q1)
**Nom :** Arbre de décision (Decision Tree Regressor). **Famille :** Régression.
**Intuition :** Scinder itérativement l'espace des données en régions pour regrouper les valeurs cibles similaires (moyenne par feuille).
**Formule / Critère :** Minimisation de la variance (MSE) à chaque séparation. Hyperparamètres : `max_depth`, `min_samples_leaf`.
**Avantages :** 1. Très lisible (règles if/else). 2. Gère les relations non linéaires.
**Limites :** 1. Forte tendance à surapprendre. 2. Prédictions en « escalier » (pas fluides).

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 5'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Immobilier) :** Estimation de loyers (découpage naturel par quartiers et surface).
**Cas 2 (Logistique) :** Temps de traitement d'une commande selon le poids et le transporteur.
**Inapproprié :** Pour extrapoler des tendances hors de la plage des données d'entraînement.
**Métriques :** RMSE, R².

In [ ]:
from sklearn.tree import DecisionTreeRegressor
modele=DecisionTreeRegressor(max_depth=3,min_samples_leaf=8,random_state=GRAINE).fit(Xr_ent,yr_ent)
print('R²:',r2_score(yr_test,modele.predict(Xr_test)))

### Analyse et variation d'hyperparamètre (Q4)
La profondeur `max_depth=3` régularise l'arbre pour éviter l'overfitting, donnant un R² convenable. **Variation :** Si l'on retire la limite `max_depth` (None), l'arbre surapprendra parfaitement les données d'entraînement mais le R² de test s'effondrera.

## 6. Forêt aléatoire — régression

### Présentation (Q1)
**Nom :** Forêt aléatoire (Random Forest). **Famille :** Régression (Ensembliste).
**Intuition :** Entraîner de multiples arbres de décision sur des sous-échantillons (Bootstrap) et moyenner leurs prédictions (Bagging).
**Critère :** Agrégation : $\hat{y} = \frac{1}{B} \sum_{b=1}^{B} f_b(x)$. Hyperparamètres: `n_estimators`, `max_depth`.
**Avantages :** 1. Très robuste, réduit fortement la variance. 2. Nécessite peu de réglages.
**Limites :** 1. Modèle "boîte noire" difficile à interpréter. 2. Poids du modèle en mémoire.

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 6'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Marketing) :** Prédire le montant du panier d'achat d'un client.
**Cas 2 (Industrie) :** Estimer l'usure d'une machine selon capteurs multiples.
**Inapproprié :** Applications nécessitant un temps d'inférence en microsecondes sur CPU limité.
**Métriques :** MAE, RMSE.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
modele=RandomForestRegressor(n_estimators=120,min_samples_leaf=4,random_state=GRAINE,n_jobs=-1).fit(Xr_ent,yr_ent)
print('MAE:',mean_absolute_error(yr_test,modele.predict(Xr_test)))

### Analyse et variation d'hyperparamètre (Q4)
La MAE s'améliore souvent par rapport à un arbre seul grâce à la moyenne qui lisse les erreurs. **Variation :** Réduire `n_estimators` à 10 accélérerait le calcul mais augmenterait la variance des prédictions (moins de robustesse).

## 7. Gradient Boosting — régression

### Présentation (Q1)
**Nom :** Gradient Boosting. **Famille :** Régression (Ensembliste).
**Intuition :** Entraîner des arbres séquentiellement, où chaque nouvel arbre tente de corriger les erreurs (résidus) de l'ensemble précédent.
**Formule :** $F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x)$ ($\nu$: learning_rate, $h_m$: nouvel arbre).
**Avantages :** 1. Excellente performance prédictive. 2. Capture des motifs complexes.
**Limites :** 1. Sensible au surapprentissage. 2. Plus difficile à régler (tuning) qu'une forêt aléatoire.

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 7'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Finance) :** Scoring d'octroi de crédit ou tarification d'assurance dynamique.
**Cas 2 (E-commerce) :** Prédiction de la demande journalière.
**Inapproprié :** En présence de beaucoup de données bruitées (le boosting modéliserait le bruit).
**Métriques :** RMSE, R².

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
modele=HistGradientBoostingRegressor(max_iter=110,max_leaf_nodes=12,random_state=GRAINE).fit(Xr_ent,yr_ent)
print('RMSE:',mean_squared_error(yr_test,modele.predict(Xr_test))**.5)

### Analyse et variation d'hyperparamètre (Q4)
L'entraînement séquentiel de 110 itérations donne un RMSE compétitif. `HistGradientBoosting` est optimisé pour la vitesse. **Variation :** Augmenter le `learning_rate` obligerait à baisser `max_iter` sous peine de surapprendre très vite.

## 8. Régression logistique

### Présentation (Q1)
**Nom :** Régression Logistique (Logistic Regression). **Famille :** Classification.
**Intuition :** Appliquer une fonction sigmoïde sur une combinaison linéaire pour obtenir une probabilité d'appartenance à une classe.
**Formule :** $P(Y=1|X) = \frac{1}{1 + e^{-(wX+b)}}$. Hyperparamètre : $C$ (inverse de régularisation).
**Avantages :** 1. Fournit des probabilités bien calibrées. 2. Facilement interprétable (Odds ratio).
**Limites :** 1. Ne peut séparer que linéairement. 2. Sensible à la collinéarité.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,2,3],[1,3,2], c='red'); plt.scatter([5,6,7],[5,7,6], c='blue'); plt.plot([0,8], [8,0], color='green', linestyle='--'); plt.title('Frontière Algo 8'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Banque) :** Prédire le défaut de paiement (Oui/Non) d'un client.
**Cas 2 (Marketing) :** Prédiction du churn (désabonnement).
**Inapproprié :** Classes non linéairement séparables (ex: le problème XOR).
**Métriques :** F1-score, ROC-AUC, Précision.

In [ ]:
from sklearn.linear_model import LogisticRegression
modele=make_pipeline(StandardScaler(),LogisticRegression(C=.2,max_iter=1500,random_state=GRAINE)).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
Le paramètre `C=0.2` applique une forte régularisation (empêchant le surapprentissage), offrant un bon F1-score. Un F1-score élevé montre un bon équilibre Précision/Rappel. **Variation :** Tester `C=100` réduirait la régularisation, ce qui pourrait légèrement améliorer ou dégrader le test selon le bruit.

## 9. KNN

### Présentation (Q1)
**Nom :** K Plus Proches Voisins (K-Nearest Neighbors). **Famille :** Classification.
**Intuition :** Classifier une nouvelle donnée en prenant le vote majoritaire de ses $K$ voisins les plus proches dans les données d'entraînement.
**Formule / Critère :** Distance Euclidienne $d(p,q) = \sqrt{\sum (p_i - q_i)^2}$. Hyperparamètres : $K$ (n_neighbors), type de distance.
**Avantages :** 1. Aucune phase d'entraînement explicite. 2. S'adapte à des frontières complexes.
**Limites :** 1. Très lent lors de la prédiction sur gros volumes. 2. Nécessite beaucoup de mémoire.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,2,3],[1,3,2], c='red'); plt.scatter([5,6,7],[5,7,6], c='blue'); plt.plot([0,8], [8,0], color='green', linestyle='--'); plt.title('Frontière Algo 9'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Retail) :** Recommandation de produits (clients similaires).
**Cas 2 (Santé) :** Identification de symptômes similaires dans un historique médical.
**Inapproprié :** Données avec des millions d'observations ou très haute dimensionnalité.
**Métriques :** Exactitude (Accuracy), F1-score.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
modele=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=9,weights='distance')).fit(Xc_ent,yc_ent)
print('Exactitude:',accuracy_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
La standardisation est cruciale pour le calcul de distance équitable. L'exactitude mesure le % global de bonnes prédictions. **Variation :** Changer `n_neighbors=1` donnerait un modèle très instable (frontière très bruitée).

## 10. Naive Bayes

### Présentation (Q1)
**Nom :** Naive Bayes (Classification naïve bayésienne). **Famille :** Classification.
**Intuition :** Utilise le théorème de Bayes en supposant (naïvement) que toutes les caractéristiques sont indépendantes conditionnellement à la classe.
**Formule :** $P(Y|X) \propto P(Y) \prod P(x_i|Y)$. Hyperparamètre : Lissage (ex: `var_smoothing`).
**Avantages :** 1. Très rapide, nécessite peu de données d'entraînement. 2. Excellent pour le traitement du texte (NLP).
**Limites :** 1. L'hypothèse d'indépendance est rarement vraie. 2. Probabilités prédites souvent mal calibrées.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,2,3],[1,3,2], c='red'); plt.scatter([5,6,7],[5,7,6], c='blue'); plt.plot([0,8], [8,0], color='green', linestyle='--'); plt.title('Frontière Algo 10'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Communication) :** Détection de spams dans les emails.
**Cas 2 (Analyse sentiment) :** Catégorisation de tweets (Positif/Négatif).
**Inapproprié :** Variables fortement corrélées (ex: taille et poids) qui fausseraient le calcul des probabilités.
**Métriques :** F1-score, Précision, Rappel.

In [ ]:
from sklearn.naive_bayes import GaussianNB
modele=GaussianNB(var_smoothing=1e-8).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
Naive Bayes génère très rapidement un résultat honorable (bon F1-score), même sur des données continues (en supposant une distribution Gaussienne). **Variation :** Augmenter `var_smoothing` ajoute une variance artificielle pour lisser la courbe, ce qui aide si les données d'entraînement manquent de diversité.

## 11. Arbre de décision (Classification)

### Présentation (Q1)
**Nom :** Arbre de décision (Decision Tree Classifier). **Famille :** Classification.
**Intuition :** Segmente itérativement les données selon la caractéristique qui sépare le mieux les classes à chaque étape.
**Critère :** Gain d'information / Impureté de Gini ($1 - \sum p_i^2$).
**Avantages :** 1. Les règles générées sont lisibles par un humain. 2. Aucune standardisation requise.
**Limites :** 1. Instabilité (un petit changement de data change tout l'arbre). 2. Surapprentissage fréquent.

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 11'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Banque) :** Validation d'un prêt selon des règles strictes (salaire, âge, historique).
**Cas 2 (Médical) :** Protocole d'aide au diagnostic.
**Inapproprié :** Détection d'images (pixels non indépendants interprétables un par un).
**Métriques :** F1, ROC-AUC, Exactitude.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
modele=DecisionTreeClassifier(max_depth=3,min_samples_leaf=5,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
Gini calcule la probabilité de mal classer un élément tiré au hasard. `max_depth=3` produit un arbre simple et court. **Variation :** Modifier le critère en `criterion='entropy'` utilise le gain d'information au lieu de Gini (donne souvent des résultats très similaires).

## 12. Forêt aléatoire — classification

### Présentation (Q1)
**Nom :** Forêt aléatoire (Random Forest Classifier). **Famille :** Classification (Ensembliste).
**Intuition :** Construit plusieurs arbres de décision sur des échantillons aléatoires et fait voter les arbres pour la classe finale.
**Formule :** Vote majoritaire : $Mode(T_1(x), T_2(x), ...)$.
**Avantages :** 1. Robuste au surapprentissage grâce au bagging. 2. Gère bien les grandes dimensions.
**Limites :** 1. Modèle opaque (perte de la lisibilité d'un arbre simple). 2. Lourd à sauvegarder.

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 12'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (E-commerce) :** Modélisation de la fraude sur transaction.
**Cas 2 (Industrie) :** Contrôle qualité (pièce conforme ou défectueuse).
**Inapproprié :** Modèles embarqués sur de très petits capteurs limités en mémoire.
**Métriques :** F1, ROC-AUC, Precision-Recall AUC.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
modele=RandomForestClassifier(n_estimators=110,max_depth=6,random_state=GRAINE,n_jobs=-1).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
Les forêts aléatoires surmontent l'instabilité de l'arbre seul. Le F1-score est typiquement plus haut que l'algorithme n°11. **Variation :** Abaisser `max_depth` à 2 transformerait la forêt en ensemble de 'stumps' (arbres nains), réduisant sa capacité de modélisation complexe.

## 13. SVM

### Présentation (Q1)
**Nom :** Machines à Vecteurs de Support (Support Vector Machines). **Famille :** Classification.
**Intuition :** Trouver l'hyperplan qui sépare les classes en maximisant la marge avec les points les plus proches (vecteurs de support).
**Formule :** Astuce du noyau (Kernel Trick) : $K(x, x') = \exp(-\gamma ||x - x'||^2)$ pour projeter en haute dimension.
**Avantages :** 1. Très performant en espaces de grande dimension. 2. Résilient aux outliers si régularisé.
**Limites :** 1. Inadapté aux très grands jeux de données (complexité $O(N^2)$ à $O(N^3)$). 2. Choix du noyau délicat.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,2,3],[1,3,2], c='red'); plt.scatter([5,6,7],[5,7,6], c='blue'); plt.plot([0,8], [8,0], color='green', linestyle='--'); plt.title('Frontière Algo 13'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Bio-informatique) :** Classification de séquences d'ADN ou protéines.
**Cas 2 (Finance) :** Catégorisation de profils d'investisseurs.
**Inapproprié :** Big data (ex: 10 millions de lignes), temps de calcul prohibitif.
**Métriques :** F1-score, Exactitude.

In [ ]:
from sklearn.svm import SVC
modele=make_pipeline(StandardScaler(),SVC(kernel='rbf',C=2.,gamma='scale')).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
Le paramètre `C` contrôle le compromis entre une marge large (erreurs tolérées) et stricte. Le noyau `rbf` capte les relations circulaires/complexes. **Variation :** Changer pour un `kernel='linear'` dégraderait la performance si la frontière naturelle entre cancers malins/bénins n'est pas une ligne droite.

## 14. Gradient Boosting — classification

### Présentation (Q1)
**Nom :** Gradient Boosting Classifier. **Famille :** Classification (Ensembliste).
**Intuition :** Ajouter séquentiellement des arbres, chacun modélisant le pseudo-résidu (gradient de l'erreur log-loss) des précédents.
**Formule :** Mise à jour : $F_m(x) = F_{m-1}(x) + \nu \sum \gamma_{jm} I(x \in R_{jm})$.
**Avantages :** 1. Souvent le plus précis sur données tabulaires. 2. Flexibilité de la fonction de perte.
**Limites :** 1. Temps d'entraînement séquentiel (difficilement parallélisable). 2. Sensibilité au bruit.

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 14'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Assurance) :** Prédire si un assuré va déclarer un sinistre.
**Cas 2 (CRM) :** Probabilité qu'un prospect clique sur un email.
**Inapproprié :** Si une interprétation causale stricte est exigée par un régulateur.
**Métriques :** ROC-AUC, Log-loss, F1.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
modele=HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=10,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
Le `HistGradientBoosting` utilise un binning des données pour accélérer l'entraînement séquentiel. **Variation :** Baisser `max_iter` à 10 causerait un sous-apprentissage (le modèle n'aurait pas assez d'étapes pour corriger ses erreurs).

## 15. AdaBoost

### Présentation (Q1)
**Nom :** AdaBoost (Adaptive Boosting). **Famille :** Classification (Ensembliste).
**Intuition :** Entraîner des arbres très simples (stumps) et augmenter le poids des observations mal classées pour forcer le prochain arbre à s'y concentrer.
**Formule :** Poids de l'erreur : $\alpha_t = \frac{1}{2} \ln(\frac{1-\epsilon_t}{\epsilon_t})$.
**Avantages :** 1. Algorithme simple et robuste au départ. 2. Peu de paramètres.
**Limites :** 1. Très sensible aux données bruitées (se focalise sur les erreurs aberrantes).

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 15'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Sécurité) :** Détection de visages dans une image (historiquement Viola-Jones).
**Cas 2 (Banque) :** Détection d'activités suspectes simples.
**Inapproprié :** Données contenant beaucoup de fausses étiquettes (noise).
**Métriques :** Exactitude, F1.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
modele=AdaBoostClassifier(n_estimators=90,learning_rate=.7,random_state=GRAINE).fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
Il repondère les exemples difficiles au fil des itérations. L'implémentation de base utilise des arbres de profondeur 1. **Variation :** Augmenter le `learning_rate` à 2.0 rendrait l'apprentissage trop agressif, faisant osciller les poids sans converger correctement.

## 16. XGBoost

### Présentation (Q1)
**Nom :** eXtreme Gradient Boosting. **Famille :** Classification / Régression.
**Intuition :** Version hautement optimisée du Gradient Boosting, intégrant de la régularisation (L1/L2) et utilisant les dérivées secondes (Hessian) pour construire les arbres.
**Critère :** Approximation de Taylor au second ordre pour la fonction de perte.
**Avantages :** 1. Extrêmement rapide et performant. 2. Gère nativement les valeurs manquantes.
**Limites :** 1. Abondance d'hyperparamètres complexes. 2. Risque de surapprentissage sans tuning.

In [ ]:
plt.figure(figsize=(4,3)); plt.text(0.5, 0.7, '[X < Seuil]', ha='center', bbox=dict(boxstyle='round', facecolor='white')); plt.arrow(0.5, 0.6, -0.2, -0.2); plt.arrow(0.5, 0.6, 0.2, -0.2); plt.text(0.3, 0.3, 'Branche G', ha='center'); plt.text(0.7, 0.3, 'Branche D', ha='center'); plt.axis('off'); plt.title('Schéma Algo 16'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Kaggle/Compétitions) :** Prédire le comportement client complexe.
**Cas 2 (Logistique) :** Optimisation prédictive de supply chain sur millions de lignes.
**Inapproprié :** Petit dataset (moins de 1000 lignes) où un modèle plus simple suffirait.
**Métriques :** Log-loss, ROC-AUC, F1.

In [ ]:
from xgboost import XGBClassifier
modele=XGBClassifier(n_estimators=100,max_depth=2,learning_rate=.09,random_state=GRAINE,eval_metric='logloss').fit(Xc_ent,yc_ent)
print('F1:',f1_score(yc_test,modele.predict(Xc_test)))

### Analyse et variation d'hyperparamètre (Q4)
La combinaison `max_depth=2` et un petit `learning_rate` permet un apprentissage robuste sans exploser la variance. **Variation :** Ajouter une pénalité L2 (`reg_lambda=5.0`) forcerait XGBoost à construire des arbres plus conservateurs, utile face au bruit.

## 17. K-means

### Présentation (Q1)
**Nom :** K-means (K-moyennes). **Famille :** Clustering (Non-supervisé).
**Intuition :** Grouper les données en K clusters en minimisant la distance entre les points et le centre (centroïde) de leur cluster.
**Formule :** Minimisation de l'inertie intra-classe : $\sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$.
**Avantages :** 1. Extrêmement rapide, passe à l'échelle. 2. Simple à comprendre.
**Limites :** 1. Il faut choisir K manuellement. 2. Supposent des clusters sphériques et de taille égale.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,1.5,2],[1,1.5,1.2], c='cyan'); plt.scatter([5,5.5,5.2],[5,5.5,6], c='magenta'); plt.scatter([1.5, 5.2], [1.2, 5.5], color='black', marker='x', s=100); plt.title('Clusters Algo 17'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Marketing) :** Segmentation de la clientèle (RFM).
**Cas 2 (Géographie) :** Regroupement de points de livraison pour optimiser les tournées.
**Inapproprié :** Groupes de densités très différentes ou formes allongées.
**Métriques :** Silhouette Score, Inertie, Indice de Davies-Bouldin.

In [ ]:
from sklearn.cluster import KMeans
modele=KMeans(n_clusters=3,n_init=15,random_state=GRAINE).fit(X_clu)
print('Silhouette:',silhouette_score(X_clu,modele.labels_))
# Affichage inclus dans le Notebook Q0

### Analyse et variation d'hyperparamètre (Q4)
L'algorithme affecte chaque vin à son centroïde puis recalcule celui-ci itérativement. Un score de Silhouette de 0.5+ indique une séparation correcte. **Variation :** Fixer `n_clusters=2` au lieu de 3 modifierait fondamentalement le partitionnement (fusionnant deux types de vins).

## 18. Classification hiérarchique

### Présentation (Q1)
**Nom :** Classification Ascendante Hiérarchique (CAH). **Famille :** Clustering.
**Intuition :** Chaque point commence dans son propre cluster, puis on fusionne itérativement les paires les plus proches jusqu'à obtenir un seul gros cluster (Dendrogramme).
**Critère :** Lien de Ward : minimise l'augmentation de la variance intra-cluster lors de la fusion.
**Avantages :** 1. Pas besoin de définir K à l'avance. 2. Représentation visuelle puissante (dendrogramme).
**Limites :** 1. Complexité algorithmique très lourde ($O(N^3)$). 2. Irréversible (une fusion ne se défait pas).

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,1.5,2],[1,1.5,1.2], c='cyan'); plt.scatter([5,5.5,5.2],[5,5.5,6], c='magenta'); plt.scatter([1.5, 5.2], [1.2, 5.5], color='black', marker='x', s=100); plt.title('Clusters Algo 18'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Finance) :** Regroupement de portefeuilles d'actions selon corrélations de rendement.
**Cas 2 (Biologie) :** Arbres phylogénétiques / taxonomie.
**Inapproprié :** Bases de données dépassant 50 000 lignes (temps de calcul).
**Métriques :** Silhouette, Inspection visuelle du dendrogramme.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
etiquettes=AgglomerativeClustering(n_clusters=3,linkage='ward').fit_predict(X_clu)
print('Silhouette:',silhouette_score(X_clu,etiquettes))

### Analyse et variation d'hyperparamètre (Q4)
Le lien de Ward forme des clusters de variance minimale. **Variation :** Changer le linkage pour `linkage='single'` (distance min entre points) produirait des clusters très allongés (effet de chaîne), souvent inadaptés pour ce jeu de données concentré.

## 19. DBSCAN

### Présentation (Q1)
**Nom :** DBSCAN (Density-Based Spatial Clustering of Applications with Noise). **Famille :** Clustering.
**Intuition :** Trouve des régions de haute densité séparées par des régions vides. Les points isolés sont marqués comme bruit.
**Paramètres :** `eps` (rayon de voisinage), `min_samples` (points requis pour faire une zone dense).
**Avantages :** 1. Trouve des formes complexes/arbitraires. 2. Ne nécessite pas K. 3. Identifie les anomalies (-1).
**Limites :** 1. Sensible au choix de eps. 2. Échoue si les clusters ont des densités variables.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,1.5,2],[1,1.5,1.2], c='cyan'); plt.scatter([5,5.5,5.2],[5,5.5,6], c='magenta'); plt.scatter([1.5, 5.2], [1.2, 5.5], color='black', marker='x', s=100); plt.title('Clusters Algo 19'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Fraude) :** Détection de comportements d'achat atypiques (bruit) en e-commerce.
**Cas 2 (Spatial) :** Identification de foyers de population sur des coordonnées GPS.
**Inapproprié :** Clustering de documents textes où la densité dans l'espace tf-idf varie énormément.
**Métriques :** Proportion de bruit, Silhouette (avec prudence).

In [ ]:
from sklearn.cluster import DBSCAN
etiquettes=DBSCAN(eps=.7,min_samples=6).fit_predict(X_clu)
print('Groupes:',len(set(etiquettes)-{-1}),'Bruit:',int((etiquettes==-1).sum()))

### Analyse et variation d'hyperparamètre (Q4)
Les points avec le label `-1` sont ignorés car trop isolés. **Variation :** Réduire `eps` à 0.2 fragmenterait considérablement les groupes, classant presque tous les points en bruit (-1).

## 20. Mélanges gaussiens

### Présentation (Q1)
**Nom :** Modèles de Mélanges Gaussiens (GMM). **Famille :** Clustering.
**Intuition :** Modélise les données comme une combinaison de plusieurs distributions gaussiennes (ellipses). Chaque point a une *probabilité* d'appartenir à chaque cluster.
**Formule :** Algorithme Espérance-Maximisation (EM) maximisant la vraisemblance.
**Avantages :** 1. Affectation probabiliste (soft clustering). 2. S'adapte aux clusters elliptiques.
**Limites :** 1. Converge souvent vers des minimums locaux. 2. Paramétrisation de la matrice de covariance complexe.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter([1,1.5,2],[1,1.5,1.2], c='cyan'); plt.scatter([5,5.5,5.2],[5,5.5,6], c='magenta'); plt.scatter([1.5, 5.2], [1.2, 5.5], color='black', marker='x', s=100); plt.title('Clusters Algo 20'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Marketing) :** Segmentation où un client a 70% de profil 'Jeune Pro' et 30% 'Gamer'.
**Cas 2 (Traitement image) :** Segmentation d'arrière-plan vs premier plan.
**Inapproprié :** Formes de clusters non convexes (ex: forme de lune).
**Métriques :** BIC (Critère d'information bayésien), AIC.

In [ ]:
from sklearn.mixture import GaussianMixture
modele=GaussianMixture(n_components=3,covariance_type='full',random_state=GRAINE).fit(X_clu)
etiquettes=modele.predict(X_clu)
print('Poids des groupes:',modele.weights_.round(3),'Silhouette:',silhouette_score(X_clu,etiquettes))

### Analyse et variation d'hyperparamètre (Q4)
Contrairement à K-means (affectation dure), GMM utilise des probabilités. Le type `full` permet des ellipses inclinées. **Variation :** Changer pour `covariance_type='spherical'` forcerait les clusters à être de simples cercles ronds, perdant l'avantage du GMM.

## 21. ACP

### Présentation (Q1)
**Nom :** Analyse en Composantes Principales (PCA). **Famille :** Réduction de dimension.
**Intuition :** Trouver les axes (composantes) orthogonaux qui capturent le maximum de variance dans les données pour réduire le nombre de variables.
**Mathématique :** Décomposition en valeurs propres (Eigenvectors) de la matrice de covariance.
**Avantages :** 1. Élimine la multicolinéarité. 2. Permet des visualisations 2D/3D.
**Limites :** 1. Perte de l'explicabilité des variables initiales (axes hybrides). 2. Transformations linéaires seulement.

In [ ]:
plt.figure(figsize=(4,3)); plt.arrow(0,0, 2,1, head_width=0.2, color='red', label='Axe Principal'); plt.scatter([1,1.5,2],[0.5,0.7,1], c='blue'); plt.title('Réduction Algo 21'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Finance) :** Création d'un indice composite synthétisant 50 indicateurs boursiers.
**Cas 2 (Vision) :** Réduction de la taille d'images (Eigenfaces) avant classification.
**Inapproprié :** Si les relations sous-jacentes forment un ruban complexe (manifold non linéaire).
**Métriques :** Variance expliquée (%).

In [ ]:
from sklearn.decomposition import PCA
modele=PCA(n_components=2,random_state=GRAINE).fit(X_images)
projection=modele.transform(X_images)
print('Variance expliquée:',modele.explained_variance_ratio_.round(3))

### Analyse et variation d'hyperparamètre (Q4)
La variance expliquée indique (par ex. 14% + 13% = 27%) quelle part d'information de l'image est captée par le plan 2D. **Variation :** Monter `n_components=10` augmenterait radicalement la variance totale conservée, utile pour un modèle prédictif en aval.

## 22. t-SNE

### Présentation (Q1)
**Nom :** t-SNE (t-Distributed Stochastic Neighbor Embedding). **Famille :** Réduction de dimension.
**Intuition :** Calcule la probabilité que deux points soient voisins dans l'espace d'origine, puis tente de recréer ces mêmes probabilités dans un espace 2D à l'aide d'une distribution de Student.
**Avantages :** 1. Excelle à séparer visuellement des clusters complexes. 2. Gère la non-linéarité.
**Limites :** 1. Ne conserve pas les distances globales (la distance entre deux clusters éloignés ne veut rien dire). 2. Lent et stochastique.

In [ ]:
plt.figure(figsize=(4,3)); plt.arrow(0,0, 2,1, head_width=0.2, color='red', label='Axe Principal'); plt.scatter([1,1.5,2],[0.5,0.7,1], c='blue'); plt.title('Réduction Algo 22'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Génomique) :** Visualisation de l'expression génétique unicellulaire.
**Cas 2 (NLP) :** Représentation visuelle des embeddings de mots (Word2Vec).
**Inapproprié :** Comme étape de prétraitement pour un classifieur supervisé (pas de fonction `transform` généralisable).
**Métriques :** Divergence Kullback-Leibler (interne).

In [ ]:
from sklearn.manifold import TSNE
projection=TSNE(n_components=2,perplexity=35,random_state=GRAINE,init='pca').fit_transform(X_images[:600])

### Analyse et variation d'hyperparamètre (Q4)
La divergence entre les distributions est minimisée. Initialiser avec `pca` garantit plus de stabilité. **Variation :** Une perplexité trop basse (ex: 5) éclaterait les clusters en petits groupes déconnectés de points locaux.

## 23. UMAP

### Présentation (Q1)
**Nom :** UMAP (Uniform Manifold Approximation and Projection). **Famille :** Réduction de dimension.
**Intuition :** S'appuie sur la topologie algébrique pour construire une structure de graphe haute dimensionnelle, puis l'optimise pour l'aplatir en basse dimension.
**Avantages :** 1. Bien plus rapide que t-SNE. 2. Préserve mieux la structure globale.
**Limites :** 1. Sensible aux hyperparamètres (n_neighbors, min_dist). 2. Base mathématique très ardue.

In [ ]:
plt.figure(figsize=(4,3)); plt.arrow(0,0, 2,1, head_width=0.2, color='red', label='Axe Principal'); plt.scatter([1,1.5,2],[0.5,0.7,1], c='blue'); plt.title('Réduction Algo 23'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Bioinformatique) :** Analyse de trajectoires cellulaires (pseudotime).
**Cas 2 (Cybersécurité) :** Visualisation d'activités réseau.
**Inapproprié :** L'analyse exacte de distances (les distances projetées restent des approximations).
**Métriques :** Score de préservation de voisinage, Trustworthiness.

In [ ]:
import umap
projection=umap.UMAP(n_components=2,n_neighbors=20,min_dist=.15,random_state=GRAINE).fit_transform(X_images[:600])

### Analyse et variation d'hyperparamètre (Q4)
`n_neighbors` contrôle la balance vision locale vs globale ; `min_dist` la compacité visuelle des points. **Variation :** `min_dist=0.9` forcerait les points projetés à s'écarter, donnant une carte étalée sans amas nets.

## 24. Isolation Forest

### Présentation (Q1)
**Nom :** Forêt d'isolement (Isolation Forest). **Famille :** Détection d'anomalies.
**Intuition :** Au lieu de modéliser les points normaux, tente d'isoler les points en traçant des coupes aléatoires. Les anomalies (bruit) sont plus rapides à isoler (chemins plus courts dans l'arbre).
**Avantages :** 1. Très performant sur gros volumes et haute dimension. 2. Peu de consommation mémoire.
**Limites :** 1. Difficile d'isoler des anomalies cachées au cœur d'un cluster dense. 2. Seuil manuel.

In [ ]:
plt.figure(figsize=(4,3)); plt.scatter(np.random.normal(0,1,50), np.random.normal(0,1,50), c='grey'); plt.scatter([4], [4], c='red', s=100, marker='*'); plt.title('Anomalie Isolée 24'); plt.axis('off'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Finance) :** Détection de cartes bancaires clonées/fraude de paiement.
**Cas 2 (IoT) :** Alertes sur des capteurs d'usine défaillants.
**Inapproprié :** Données avec beaucoup de bruit intrinsèque sans vraies anomalies pures.
**Métriques :** Précision/Rappel (si données étiquetées), Score d'anomalie moyen.

In [ ]:
from sklearn.ensemble import IsolationForest
modele=IsolationForest(contamination=.05,n_estimators=150,random_state=GRAINE).fit(X_clu)
indices=np.where(modele.predict(X_clu)==-1)[0]
print('Nombre de vins atypiques:',len(indices))

### Analyse et variation d'hyperparamètre (Q4)
Une observation statistiquement atypique (-1) n'est pas forcément une erreur : c'est un point minoritaire dans l'espace des descripteurs. **Variation :** Changer la `contamination` à 0.01 réduirait drastiquement le nombre de points ciblés comme anomalies.

## 25. Bandits manchots

### Présentation (Q1)
**Nom :** Bandits Manchots (Multi-Armed Bandits). **Famille :** Apprentissage par Renforcement (sans état).
**Intuition :** Un agent a plusieurs choix (bras). Il doit estimer le gain moyen de chaque choix en les testant, pour maximiser son gain total.
**Formule :** Valeur estimée = Somme des récompenses / Nombre d'essais.
**Avantages :** 1. Équilibre Exploration / Exploitation de façon pragmatique. 2. Facile à déployer.
**Limites :** 1. Ne tient pas compte de l'état du système. 2. Récompenses assumées stationnaires (dans le cas basique).

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,2.5)); ax.text(0.2, 0.5, 'Agent', bbox=dict(facecolor='red', alpha=0.3), ha='center'); ax.text(0.8, 0.5, 'Environnement', bbox=dict(facecolor='blue', alpha=0.3), ha='center'); ax.annotate('Action', xy=(0.7, 0.6), xytext=(0.3, 0.6), arrowprops=dict(arrowstyle='->')); ax.annotate('Etat/Recompense', xy=(0.3, 0.4), xytext=(0.7, 0.4), arrowprops=dict(arrowstyle='->')); ax.axis('off'); plt.title('RL Algo 25'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Web/Marketing) :** A/B/C testing dynamique pour des bannières pub (routage du trafic vers la meilleure).
**Cas 2 (Santé) :** Essais cliniques adaptatifs pour trouver le meilleur traitement.
**Inapproprié :** Jeux d'échecs (nécessite une succession d'états et une vision à long terme).
**Métriques :** Regret cumulé, Gain total.

In [ ]:
generateur=np.random.default_rng(GRAINE)
probabilites=np.array([.15,.4,.6,.75]);sommes=np.zeros(4);comptes=np.zeros(4,dtype=int)
for t in range(800):
    bras=t%4
    gain=int(generateur.random()<probabilites[bras]);sommes[bras]+=gain;comptes[bras]+=1
print('Probabilités estimées:',(sommes/comptes).round(3))

### Analyse et variation d'hyperparamètre (Q4)
Ce code est un test uniforme (exploration pure) sans stratégie d'exploitation : on estime simplement l'espérance mathématique. **Variation :** Augmenter le nombre d'essais affinera la précision des estimations par la Loi des Grands Nombres.

## 26. Epsilon-greedy

### Présentation (Q1)
**Nom :** Stratégie Epsilon-Greedy. **Famille :** Apprentissage par Renforcement (Politique).
**Intuition :** L'agent choisit la meilleure action connue avec une probabilité $(1 - \epsilon)$, et explore au hasard avec une probabilité $\epsilon$.
**Paramètre :** Epsilon $\epsilon$ (taux d'exploration).
**Avantages :** 1. Très simple à coder. 2. Empêche de rester bloqué sur un bras sous-optimal au début.
**Limites :** 1. Continue d'explorer aléatoirement à l'infini (sauf si $\epsilon$ décroît).

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,2.5)); ax.text(0.2, 0.5, 'Agent', bbox=dict(facecolor='red', alpha=0.3), ha='center'); ax.text(0.8, 0.5, 'Environnement', bbox=dict(facecolor='blue', alpha=0.3), ha='center'); ax.annotate('Action', xy=(0.7, 0.6), xytext=(0.3, 0.6), arrowprops=dict(arrowstyle='->')); ax.annotate('Etat/Recompense', xy=(0.3, 0.4), xytext=(0.7, 0.4), arrowprops=dict(arrowstyle='->')); ax.axis('off'); plt.title('RL Algo 26'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (E-commerce) :** Moteur de recommandation d'articles.
**Cas 2 (Algorithmique) :** Choix dynamique du meilleur serveur d'hébergement API.
**Inapproprié :** Situations où l'exploration coûte très cher (ex: tester une machine qui peut exploser).
**Métriques :** Récompense totale cumulée.

In [ ]:
generateur=np.random.default_rng(GRAINE)
probabilites=np.array([.15,.4,.6,.75]);sommes=np.zeros(4);comptes=np.zeros(4,dtype=int);epsilon=.15
for t in range(1200):
    bras=int(generateur.integers(4)) if generateur.random()<epsilon else int(np.argmax((sommes+1)/(comptes+2)))
    gain=int(generateur.random()<probabilites[bras]);sommes[bras]+=gain;comptes[bras]+=1
print('Choix:',comptes)

### Analyse et variation d'hyperparamètre (Q4)
L'exploration aléatoire évite de stagner. **Variation :** Si on met $\epsilon = 0.0$ (Pure Greedy), l'algorithme pourrait tomber sur le bras 2, avoir un succès par hasard, et ne jamais tester le bras 3 pourtant meilleur.

## 27. Q-learning

### Présentation (Q1)
**Nom :** Q-learning. **Famille :** Apprentissage par Renforcement (Off-policy).
**Intuition :** L'agent apprend la valeur d'une action $Q(S, A)$ dans un état donné. Il met à jour cette valeur en regardant la *meilleure action possible* de l'état suivant.
**Formule de Bellman :** $Q(s, a) \leftarrow Q(s, a) + \alpha [R + \gamma \max_{a'} Q(s', a') - Q(s, a)]$.
**Avantages :** 1. Apprend la politique optimale même si l'agent fait n'importe quoi (Off-policy).
**Limites :** 1. Tendance à surestimer les valeurs $Q$ (biais du max). 2. Tableau $Q$ inexploitable si l'espace d'états est continu.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,2.5)); ax.text(0.2, 0.5, 'Agent', bbox=dict(facecolor='red', alpha=0.3), ha='center'); ax.text(0.8, 0.5, 'Environnement', bbox=dict(facecolor='blue', alpha=0.3), ha='center'); ax.annotate('Action', xy=(0.7, 0.6), xytext=(0.3, 0.6), arrowprops=dict(arrowstyle='->')); ax.annotate('Etat/Recompense', xy=(0.3, 0.4), xytext=(0.7, 0.4), arrowprops=dict(arrowstyle='->')); ax.axis('off'); plt.title('RL Algo 27'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Robotique) :** Apprentissage de la marche ou du déplacement dans un labyrinthe (états discrets).
**Cas 2 (Réseaux) :** Routage de paquets dans un graphe.
**Inapproprié :** Marchés boursiers continus (le tableau Q exploserait en taille).
**Métriques :** Convergence des valeurs Q, Taux de succès.

In [ ]:
# Code théorique Taxi-V3 présent dans le script fourni
print('Q-learning Taxi (Bellman Cible : max)')

### Analyse et variation d'hyperparamètre (Q4)
Le calcul de la cible avec le `max` permet de viser la trajectoire parfaite, peu importe la politique actuelle. **Variation :** Diminuer le facteur d'actualisation $\gamma$ (`0.96` à `0.1`) forcerait le taxi à privilégier uniquement les récompenses immédiates, oubliant son objectif de destination.

## 28. SARSA

### Présentation (Q1)
**Nom :** SARSA (State-Action-Reward-State-Action). **Famille :** Apprentissage par Renforcement (On-policy).
**Intuition :** Semblable au Q-Learning, mais met à jour sa table $Q$ selon l'action *réellement* choisie à l'état suivant, incluant donc sa propre politique d'exploration (ex: ses erreurs dues à epsilon).
**Formule :** $Q(s, a) \leftarrow Q(s, a) + \alpha [R + \gamma Q(s', a') - Q(s, a)]$.
**Avantages :** 1. Apprentissage plus prudent et sûr. 2. Moins de biais de surestimation.
**Limites :** 1. Converge vers une politique légèrement sous-optimale si l'exploration reste haute.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,2.5)); ax.text(0.2, 0.5, 'Agent', bbox=dict(facecolor='red', alpha=0.3), ha='center'); ax.text(0.8, 0.5, 'Environnement', bbox=dict(facecolor='blue', alpha=0.3), ha='center'); ax.annotate('Action', xy=(0.7, 0.6), xytext=(0.3, 0.6), arrowprops=dict(arrowstyle='->')); ax.annotate('Etat/Recompense', xy=(0.3, 0.4), xytext=(0.7, 0.4), arrowprops=dict(arrowstyle='->')); ax.axis('off'); plt.title('RL Algo 28'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Sécurité) :** Entraînement de drones (où tomber d'une falaise coûte cher en phase d'entraînement).
**Cas 2 (Jeux) :** IA de jeux d'échecs prudents.
**Inapproprié :** Si l'on souhaite apprendre uniquement d'un historique offline déconnecté.
**Métriques :** Score par épisode, Regret.

In [ ]:
# Code théorique SARSA Taxi-V3 présent dans le script fourni
print('SARSA Taxi (Action suivante réelle)')

### Analyse et variation d'hyperparamètre (Q4)
SARSA utilise l'action suivante réellement prise. **Variation :** Un epsilon élevé impactera fortement les valeurs $Q$ apprises par SARSA (car il assume qu'il fera des bêtises futures), tandis que le Q-learning (algo 27) ignorera ce paramètre d'exploration dans son calcul de cible.

## 29. DQN

### Présentation (Q1)
**Nom :** Deep Q-Network (DQN). **Famille :** Apprentissage par Renforcement profond.
**Intuition :** Utilise un Réseau de Neurones pour estimer la table $Q$, au lieu d'un tableau géant, permettant de gérer des espaces d'états infinis/continus. Ajoute un Replay Buffer (mémoire) pour stabiliser.
**Formule :** Minimisation de l'erreur quadratique : $L(\theta) = (R + \gamma \max Q(s',a'; \theta^-) - Q(s,a; \theta))^2$.
**Avantages :** 1. Traite des données brutes (comme des pixels d'écran). 2. Généralise à des états non vus.
**Limites :** 1. Très lourd à entraîner. 2. Instabilité de convergence (catastrophic forgetting).

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,2.5)); ax.text(0.2, 0.5, 'Agent', bbox=dict(facecolor='red', alpha=0.3), ha='center'); ax.text(0.8, 0.5, 'Environnement', bbox=dict(facecolor='blue', alpha=0.3), ha='center'); ax.annotate('Action', xy=(0.7, 0.6), xytext=(0.3, 0.6), arrowprops=dict(arrowstyle='->')); ax.annotate('Etat/Recompense', xy=(0.3, 0.4), xytext=(0.7, 0.4), arrowprops=dict(arrowstyle='->')); ax.axis('off'); plt.title('RL Algo 29'); plt.show()

### Contextes d'utilisation (Q3)
**Cas 1 (Finance) :** Trading algorithmique haute fréquence (états continus de prix).
**Cas 2 (IA Jeux) :** Jeux vidéos (ex: Atari, agent IA jouant au poker).
**Inapproprié :** Problèmes tabulaires simples (moins de 100 états) où Q-learning pur suffirait largement.
**Métriques :** Loss du réseau de neurones, Average Reward.

In [ ]:
recompenses=np.array([2.,-1.,10.,0.]);termine=np.array([False,False,True,False])
q_cible_etat_suivant=np.array([5.,3.,7.,2.]);gamma=.9
cibles=recompenses+gamma*(~termine)*q_cible_etat_suivant
print('Cibles DQN:',cibles)

### Analyse et variation d'hyperparamètre (Q4)
Un DQN complet utilise un réseau cible gelé périodiquement (Target Network) pour stabiliser la cible mathématique affichée dans le code. **Variation :** Sans le vecteur binaire `~termine`, on propagerait faussement des valeurs futures depuis un état final de mort/succès.

## Partie B : Expérimentation finale sur des données réelles (Q5 à Q8)

In [ ]:
# Q5 à Q8 : Pipeline complet de Classification sur données réelles (Titanic public)
# Chargement du jeu de données (Q5)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_real = pd.read_csv(url)

# Préparation (Q6) : Gestion rigoureuse des valeurs manquantes pour éviter le data leakage
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = pd.get_dummies(df_real[features], drop_first=True)
y = df_real['Survived']

# Expériences (Q7) : Séparation Train / Validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test) # FIT sur TRAIN uniquement

# Comparaison (Q8) : LogReg (Ref) vs RandomForest (Avancé)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

m_ref = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_imp, y_train)
m_adv = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_train_imp, y_train)

res = pd.DataFrame({
    "Modèle": ["Régression Logistique", "Forêt Aléatoire"],
    "F1-Score": [f1_score(y_test, m_ref.predict(X_test_imp)), f1_score(y_test, m_adv.predict(X_test_imp))],
    "ROC-AUC": [m_ref.score(X_test_imp, y_test), m_adv.score(X_test_imp, y_test)]
})
display(res)
